# HW4 Part 2 (Bonus) — Interactive AWGN Communication System
**CS515 Deep Learning | Sabanci University**

This notebook:
1. Clones the repo and verifies the environment
2. Runs pre-training smoke tests to validate all components
3. Trains the end-to-end TX + RX communication system
4. Evaluates over all 4096 messages × 100 noise trials
5. Displays metrics and plots, then downloads results

> **Important:** Runtime → Change runtime type → **T4 GPU** must be selected.

## 0. GPU Check

In [ ]:
import torch

print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — select T4 GPU from the Runtime menu.')

## 1. Repo Setup

In [ ]:
import os

REPO_URL = 'https://github.com/caltinuzengi/cs515-deep-learning.git'
REPO_DIR = 'cs515-deep-learning'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print('Repo already exists, pulling latest changes...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 2. Environment Check

The bonus experiment requires only PyTorch and matplotlib, both pre-installed in Colab.

In [ ]:
import torch
import matplotlib

print(f'torch      : {torch.__version__}')
print(f'matplotlib : {matplotlib.__version__}')
print(f'CUDA       : {torch.cuda.is_available()}')

## 3. Pre-training Smoke Tests

Validates all five components before any training starts:
- **Forward pass shapes** — logits `(B, 4, 8)`, x_t `(B, 4)`
- **Power constraint** — `||x_t||₂ ≤ 1.0` per sample
- **Initial loss** — near random baseline `ln(8) ≈ 2.079`
- **Initial accuracy** — near random baseline `1/8 = 0.125`
- **Gradient flow** — gradients reach both TXEncoder and RXDecoder

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from models.TXEncoder import TXEncoder
from models.RXDecoder import RXDecoder
from bonus.comm_system import CommSystem
from bonus.experiment_bonus import run_smoke_tests
from bonus.config_bonus import (
    DEVICE, D_EMB, D_MODEL, N_HEADS, N_LAYERS, FFN_DIM, DROPOUT,
    T_ROUNDS, N_SYMBOLS, ALPHABET, SIGMA_SQ, EPS,
)

enc  = TXEncoder(D_EMB, D_MODEL, N_HEADS, N_LAYERS, FFN_DIM, DROPOUT,
                 T_ROUNDS, N_SYMBOLS, ALPHABET, EPS).to(DEVICE)
dec  = RXDecoder(T_ROUNDS, D_MODEL, N_HEADS, N_LAYERS, FFN_DIM, DROPOUT,
                 N_SYMBOLS, ALPHABET).to(DEVICE)
comm = CommSystem(enc, dec, SIGMA_SQ, T_ROUNDS)

run_smoke_tests(comm, enc, dec, DEVICE)

## 4. Training

Trains the end-to-end communication system:
- **TX (TXEncoder):** Transformer-based encoder, runs at every round
- **RX (RXDecoder):** Transformer-based decoder, runs once after all rounds
- **Channel:** AWGN with σ² = 0.25, noiseless feedback relay
- **Optimizer:** AdamW, early stopping with patience = 20

Best checkpoint is saved to `results/bonus/checkpoints/best_model.pt`.

In [ ]:
!python bonus/experiment_bonus.py

## 5. Results — Evaluation Metrics

Full evaluation over all **4096 messages × 100 noise trials**.

In [ ]:
import json

with open('results/bonus/metrics/eval_results.json') as f:
    metrics = json.load(f)

print('=' * 45)
print('  Full Evaluation Results')
print('=' * 45)
labels = {
    'ce_loss':          'CE Loss',
    'symbol_accuracy':  'Symbol Accuracy',
    'message_accuracy': 'Message Accuracy',
    'SER':              'Symbol Error Rate (SER)',
    'MER':              'Message Error Rate (MER)',
    'avg_power':        'Avg Power (≤ 1.0)',
}
for key, label in labels.items():
    print(f'  {label:<26}: {metrics[key]:.6f}')
print('=' * 45)

## 6. Results — Training Curves

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_files = {
    'results/bonus/plots/loss_curve.png':     'Loss Curve',
    'results/bonus/plots/accuracy_curve.png': 'Accuracy Curve',
    'results/bonus/plots/power_curve.png':    'Power Curve',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (path, title) in zip(axes, plot_files.items()):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Download Results

Downloads metrics and plots as a zip archive. Checkpoints are excluded due to size.

In [ ]:
import shutil
from google.colab import files

tmp_dir = '/tmp/bonus_results'
os.makedirs(tmp_dir, exist_ok=True)
for sub in ('metrics', 'plots'):
    shutil.copytree(f'results/bonus/{sub}', f'{tmp_dir}/{sub}', dirs_exist_ok=True)

zip_path = '/tmp/bonus_results'
shutil.make_archive(zip_path, 'zip', tmp_dir)
print('bonus_results.zip ready, downloading...')
files.download(zip_path + '.zip')